# Lora 기법을 활용한 PEFT

LLM과 같은 대규모 모델에서 더욱 효율 적인 미세조정 즉, PEFT(Parameter-Efficient Fine-Tuning)을 하기위한 가장 대표적인 방식이 바로 Lora 방식 입니다.

해당 예시에서는 허깅페이스로 모델을 불러온뒤 LoraConfig를 활용하여 구글의 gemma3 모델을 한국어 뉴스 요약 테스크로 PEFT 진행하는 과정을 살펴봅니다.



## 초기화 과정

#### 라이브러리 설치

1. `transformers > 4.50.1` : gemma3 가 포함된 최신 transformers 버전
2. `bitsandbytes` : 양자화를 위한 라이브러리
3. `trl` : SFTTrainer를 활용하여 학습하기 위한 라이브러리
4. `datasets` :  Hugging Face Hub에 저장된 데이터세트를 불러오기 위한 라이브러리

다음 라이브러리 설치후 세션 재시작


In [ ]:
# =============================================================================
# [Step 1: Gemma 3 모델 로드 및 4비트 양자화(QLoRA) 준비]
# =============================================================================
#
# 1. 핵심 개념 (Core Concept):
#    - Gemma 3: 구글이 만든 최신 고성능 AI 모델입니다. (텍스트, 이미지 다루는 멀티모달 능력이 있음)
#    - Quantization (양자화):
#      -> 원래 AI 모델의 숫자들은 32비트(float32)나 16비트(float16)라는 큰 그릇에 담겨 있습니다.
#      -> 이걸 그대로 가져오면 GPU 메모리가 부족합니다.
#      -> 그래서 4비트(int4)라는 아주 작은 그릇에 옮겨 담습니다. 용량은 1/4로 줄지만 지능은 거의 유지됩니다.
#      -> 이 기술을 QLoRA(Quantized LoRA)라고 부르며, 실습의 핵심입니다.
#
# 2. 전체 흐름 (Flow):
#    ① 라이브러리 설치: bitsandbytes(압축 도구) 등 설치.
#    ② 로그인: Gemma는 허락받은 사람만 쓸 수 있어서 허깅페이스 로그인이 필수.
#    ③ 압축 설정(Config): "4비트로 압축하되, 성능은 최대한 지켜줘"라고 설정.
#    ④ 모델 로드: 설정대로 압축된 모델을 GPU 메모리에 올림.
#
# =============================================================================

# -----------------------------------------------------------------------------
# 1. 필수 라이브러리 설치
# -----------------------------------------------------------------------------
# bitsandbytes: **오늘의 핵심**. 모델을 4비트로 압축하고 가속해주는 라이브러리.
# trl, transformers, datasets: 이전 실습과 동일 (학습, 모델 로드, 데이터 로드용)
# -----------------------------------------------------------------------------
!pip install "transformers" "bitsandbytes" "trl" "datasets" "evaluate" "matplotlib-venn"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00


허깅페이스 로그인

In [ ]:
# -----------------------------------------------------------------------------
# 2. 허깅페이스 로그인 (Gated Model 접근 권한)
# -----------------------------------------------------------------------------
# Gemma 모델은 'Gated Model'이라서 허깅페이스 웹사이트에서 '사용 동의' 버튼을 눌러야 합니다.
# 그 후 발급받은 토큰으로 로그인을 해야 다운로드가 가능합니다.
# -----------------------------------------------------------------------------
from huggingface_hub import notebook_login
notebook_login()

# 1. Gemma 모델 PEFT

[Gemma3](https://developers.googleblog.com/en/gemma-explained-whats-new-in-gemma-3/) 모델은 구글에서 Gemini 모델을 만드는 데 사용된 것과 동일한 연구 및 기술로 구축한 GPT 기반의 최신 텍스트 생성 모델입니다.

Gemma3 모델은 멀티모달로, 텍스트와 이미지 입력을 처리하고 텍스트 출력을 생성가능 합니다. 예시에서는 이미지 입력은 활용하지 않고 텍스트 입력만을 활용하여 문장 요약에 최적화된 모델로 미세조정 합니다.

먼저 모델을 가져오기 전에 [Gemma3](https://huggingface.co/google/gemma-3-1b-it)에서 **라이센스 동의** 후, 로그인 해주어야 합니다.

**Gemma 모델 설명**


<img src="https://drive.google.com/uc?export=view&id=1Kr2S_7ufYiiW9uC84OgSWKNfMX12qLhf" width=500 />


https://huggingface.co/models?sort=trending&search=gemma

https://www.youtube.com/watch?v=qcjrduz_YS8


### 양자화 하여 모델 로드

LLM 모델을 더욱 특정 테스크에 최적화하여 더욱 효율 적으로 사용하기 위해 모델의 크기를 줄이는 **경량화** 작업을 진행할 수 있습니다.  

모델의 경령화 작업중 가장 간단하고 효율적인 방법중 하나가 사용 비트가 큰 실수(float64)타입으로 저장되는 파라미터를 작은 비트의 정수(int4, int8)등으로 줄이는 **양자화** 입니다.

예) "32비트 숫자를 4비트 숫자로 압축해서 메모리 절약"

In [ ]:
# -----------------------------------------------------------------------------
# 3. 모델 로드 및 양자화 설정 (이 코드가 제일 중요합니다)
# -----------------------------------------------------------------------------
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

# 사용할 모델: 구글의 Gemma 3 (10억 파라미터, 지시어 튜닝 버전)
model_id = "google/gemma-3-1b-it"

#
# 32비트(큰 상자) 데이터를 4비트(작은 상자)로 옮겨 담는 설계도입니다.

# [4비트 양자화 설정]
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,              # 핵심: "모델을 4비트로 변환해서 로딩해라" (메모리 절약)
    bnb_4bit_compute_dtype=torch.float16, # 계산할 때 자료형:
                                          # 저장(창고)은 4비트로 하지만, 꺼내서 계산(요리)할 때는 16비트로 해라.
                                          # (4비트는 계산하기엔 너무 정밀도가 떨어져서, 순간적으로만 복원해서 씁니다)
    bnb_4bit_quant_type="nf4",      # Normal Float 4:
                                    # AI 모델의 가중치는 종 모양(정규분포)으로 퍼져 있는데,
                                    # 이 모양에 최적화된 특수한 4비트 포맷을 씁니다. (성능 손실 최소화)
    bnb_4bit_use_double_quant=True  # 이중 양자화:
                                    # "양자화를 할 때 생기는 '관리표(Scale Factor)'마저도 양자화해라."
                                    # 메모리를 티끌까지 모아 절약하겠다는 뜻입니다.
)

# [모델 실제로 불러오기]
# AutoModelForCausalLM: "다음 단어 예측하기(Causal Language Modeling)" 전문 모델 로더
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config, # 위에서 만든 압축 설계도 적용
    attn_implementation="eager"              # 어텐션 계산 방식 설정 (호환성을 위해 eager 모드 사용)
).eval() # .eval(): "지금은 학습 모드가 아니야" (Dropout 등을 끕니다)

# [토크나이저 로드]
# Gemma가 알아듣는 언어 사전(Tokenizer)을 가져옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# -----------------------------------------------------------------------------
# 4. 모델 구조 확인 (압축이 잘 됐나?)
# -----------------------------------------------------------------------------
print(model)

# =============================================================================
# [출력 결과 해석 (Output Analysis)]
# =============================================================================
# 출력된 내용을 보면 모델의 뼈대가 보입니다.
#
# Gemma3ForCausalLM(
#   (model): Gemma3TextModel(
#     (embed_tokens): ...
#     (layers): ModuleList(
#       (0-25): 26 x Gemma3DecoderLayer(  -> 총 26층짜리 아파트(레이어)입니다.
#         (self_attn): Gemma3Attention(
#           (q_proj): Linear4bit(...)     -> **여기를 보세요!**
#           (k_proj): Linear4bit(...)
#
# 설명:
# 원래는 그냥 'Linear'라고 적혀 있어야 할 자리에 **'Linear4bit'**라고 적혀 있습니다.
# 이는 모델이 성공적으로 4비트로 압축되어 로딩되었다는 증거입니다.
# =============================================================================

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear4bit(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear4bit(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear4bit(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear4bit(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear4bit(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernor

### 텍스트 생성해보기

양자화된 모델로 부터 텍스트가 잘 생성되는지 테스트해 봅니다.  

`gemma-3-1b-it` 모델은 챗봇 역할로써 한번 미세조정이 된 모델이므로 입력 프롬프트 템플릿에 맞춰 텍스트를 입력해주어 좀더 명확하게 사용할 수 있습니다.

AutoTokenizer로 토크나이져를 생성한 후 `apply_chat_template()` 함수를 통해 역할이 지정된 프롬프트 텍스트를 토큰화 하고 모델 입력에 맞게 구성해 줍니다.

In [ ]:
# =============================================================================
# [Step 2: 채팅 템플릿 적용 및 입력 데이터 생성]
# =============================================================================
#
# 1. 개념 (Concept):
#    - 예전 모델(BERT 등)은 그냥 문장을 넣었지만, 최신 LLM(Gemma, ChatGPT 등)은 '채팅' 형식을 씁니다.
#    - 채팅 형식: [{"role": "user", "content": "질문"}, {"role": "assistant", "content": "답변"}]
#    - apply_chat_template: 이 채팅 리스트를 모델이 알아듣는 하나의 긴 문자열로 본드로 붙이듯 합쳐줍니다.
#      (예: "<start_of_turn>user\n질문<end_of_turn><start_of_turn>model\n")
#
# 2. 경고 메시지 해석 (Warning Analysis):
#    - "Attempting to cast a BatchEncoding to type torch.bfloat16. This is not supported."
#    - 해석: "입력 데이터(단어 번호 1, 2, 3...)는 정수여야 하는데 왜 소수점(bfloat16)으로 바꾸려 하니? 난 지원 안 해."
#    - 상황: .to(torch.bfloat16) 코드가 입력 데이터(input_ids)에는 적용되면 안 되는데 작성되어 있어서 뜬 경고입니다.
#    - 결론: 라이브러리가 똑똑해서 이 명령을 무시했습니다. 데이터는 정상적으로 GPU에 올라갔으니 괜찮습니다.
#
# =============================================================================

# -----------------------------------------------------------------------------
# 1. 토크나이저 다시 로드 (혹시 모를 초기화)
# -----------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(model_id)

# -----------------------------------------------------------------------------
# 2. 메시지 구성 (프롬프트 엔지니어링)
# -----------------------------------------------------------------------------
# 리스트 안에 리스트가 있는 구조입니다. (배치 처리를 위해)
messages = [
    [
        # 시스템 프롬프트: AI의 페르소나(역할)나 규칙을 정해줍니다.
        {
            "role": "system",
            "content": "마크다운 표기 없이 일반 텍스트 형식으로 답변 작성"
        },
        # 사용자 프롬프트: 실제 우리가 할 질문입니다.
        {
            "role": "user",
            "content": "LLM과 허깅페이스에 대한 간략한 설명"
        },
    ],
]

# -----------------------------------------------------------------------------
# 3. 템플릿 적용 및 텐서 변환 (여기가 핵심)
# -----------------------------------------------------------------------------
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True, # 마지막에 "model이 답변할 차례야"라는 신호(<start_of_turn>model)를 붙여줍니다.
    tokenize=True,              # 글자를 바로 숫자로 변환합니다.
    return_dict=True,           # 결과를 딕셔너리 형태 {'input_ids': ..., 'attention_mask': ...}로 받습니다.
    return_tensors="pt"         # 파이토치 텐서(pt)로 받습니다.
).to(model.device).to(torch.bfloat16)
# .to(model.device): 데이터를 CPU에서 GPU로 옮깁니다. (필수!)
# .to(torch.bfloat16): 여기서 경고가 발생했습니다. 입력값(input_ids)은 정수여야 하므로 사실 이 부분은 없어도 됩니다.

# -----------------------------------------------------------------------------
# 4. 결과 확인
# -----------------------------------------------------------------------------
inputs
# 출력 결과를 보면:
# 'input_ids': tensor([[ 2, 105, ... ]], device='cuda:0')
# -> 숫자들이 잘 들어있고, device='cuda:0'으로 되어 있으니 GPU에 잘 올라가 있습니다.
# -> 준비 완료입니다!

Attempting to cast a BatchEncoding to type torch.bfloat16. This is not supported.


{'input_ids': tensor([[     2,    105,   2364,    107, 238098, 238572, 164809,  41176, 237351,
          89551,  75834, 236743, 243007,  34718, 184412,   7246, 231757, 107854,
            108,   2182, 236792, 237842,  92475, 246489, 204451, 237223,  32102,
          43255, 241866, 237384,  59587,    106,    107,    105,   4368,    107]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

`generate()` 함수를 통해 텍스트를 생성해 봅니다.   
inputs에 배치 형상이 포함되었기 때문에 `batch_decode()` 함수로 디코딩을 합니다.

In [ ]:
# =============================================================================
# [Step 3: 답변 생성 (Generation) 및 해독 (Decoding)]
# =============================================================================
#
# 1. 개념 (Concept):
#    - Generate: 모델은 입력된 숫자들 뒤에 올 확률이 가장 높은 숫자를 하나씩 이어 붙입니다.
#    - Decode: 생성된 숫자 배열(List of IDs)을 다시 사람의 언어(String)로 바꿉니다.
#
# 2. 코드 설명 (Line by Line):
# =============================================================================

# -----------------------------------------------------------------------------
# 1. 생성 (Generate) - 모델이 글쓰기 시작
# -----------------------------------------------------------------------------
# with torch.inference_mode():
# -> "지금은 학습(공부) 시간이 아니라 시험(추론) 시간이야."
# -> 불필요한 연산(Gradient 계산)을 꺼서 속도를 높이고 메모리를 아낍니다.
with torch.inference_mode():
    outputs = model.generate(
        **inputs,             # 아까 만든 입력 데이터(input_ids, attention_mask)를 딕셔너리 압축 해제해서 넣습니다.

        max_new_tokens=150,   # "최대 150단어(토큰)까지만 말해." (너무 길게 말하는 것 방지)

        no_repeat_ngram_size=2,
                              # "같은 단어 쌍을 반복하지 마."
                              # 예: "사과는 맛있다. 사과는 맛있다." -> 이런 앵무새 같은 말을 막습니다.
                              # n-gram이 2니까 2단어 연속 반복 금지입니다.

        early_stopping=True   # "할 말이 다 끝났으면(EOS 토큰이 나오면) 150단어 안 채웠어도 멈춰."
    )

# -----------------------------------------------------------------------------
# 2. 해독 (Decode) - 숫자를 글자로
# -----------------------------------------------------------------------------
# tokenizer.batch_decode(...):
# -> 모델이 뱉어낸 숫자 덩어리(outputs)를 글자로 번역합니다.
outputs = tokenizer.batch_decode(
    outputs,
    skip_special_tokens=False # 중요!
                              # True: <bos>, <eos> 같은 기계용 신호를 지우고 깔끔한 문장만 보여줌.
                              # False: 기계용 신호까지 전부 다 보여줌. (지금은 이걸 썼습니다)
)

# 첫 번째 결과 출력
print(outputs[0])

# =============================================================================
# [생성 결과 분석 (Output Analysis)]
# =============================================================================
# 출력된 텍스트:
# <bos><start_of_turn>user ... <end_of_turn> <start_of_turn>model ...
#
# 1. 특수 토큰의 의미:
#    - <bos>: 문장의 시작 (Beginning of Sentence)
#    - <start_of_turn>user: "여기서부터는 사용자가 말한 거야"
#    - <end_of_turn>: "여기서 턴 종료"
#    - <start_of_turn>model: "여기서부터는 AI가 말할 차례야"
#    -> Gemma는 이 토큰들을 보고 누가 말할 차례인지 눈치채고 답변을 시작한 겁니다.
#
# 2. 시스템 프롬프트 무시 이슈:
#    - 우리는 처음에 "마크다운 표기 없이 쓰라"고 시스템 프롬프트를 줬습니다.
#    - 하지만 결과에는 '## LLM...', '**정의:**' 처럼 마크다운이 잔뜩 들어갔습니다.
#    - 이유:
#      ① 1B(10억) 모델은 체구가 작아서 지시를 완벽하게 따르지 못할 때가 있습니다.
#      ② 학습 데이터 자체가 마크다운 형식을 선호해서 습관처럼 튀어나온 것입니다.
#      ③ 이후에 할 **Fine-Tuning(미세조정)**이 바로 이런 말을 잘 듣게 교정하는 과정입니다.
# =============================================================================

<bos><start_of_turn>user
마크다운 표기 없이 일반 텍스트 형식으로 답변 작성

LLM과 허깅페이스에 대한 간략한 설명<end_of_turn>
<start_of_turn>model
## LLM(Large Language Model)과 Hugging Face에 대해 간단히 설명

**LLMs (Large language Models)이란?**

*   **정의:** 방대한 양의 훈련 데이터를 기반으로 학습된 인공지능 모델로, 인간과 유사한 language(언어)를 생성하고 이해할 수 있습니다. 
* **기능:** 챗봇처럼 다양한 종류의 질문에 답변하거나, 톤앤매너를 조절하여 글을 작성하거나 번역하는 등, 다양한 작업들을 수행할 수도 있습니다.(텍st로 작성)
    *  **예시:** GPT-3, BERT, Llama 등 유명 LLMs이 있습니다 (다양한 종류


## 요약 데이터세트 구성

이번 예시에서는 요약 테스크로 미세조정을 하기위해 허깅페이스 Dataset 허브의 [naver-news-summarization-ko]https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko 데이터를 `datasets` 라이브러리를 통해 불러오겠습니다.

load_dataset으로 로딩
허깅페이스 Dataset 허브에 업로드된 데이터는 datasets의 load_dataset() 함수를 통해 쉽게 가져 올 수 있습니다.



In [ ]:
# =============================================================================
# [Step 4: 데이터셋 로드 - "뉴스 요약"을 가르칠 교과서 준비]
# =============================================================================
#
# 1. 개념 (Concept):
#    - 우리가 하려는 건 '뉴스 기사(document)'를 보여주면 '요약문(summary)'을 뱉어내는 훈련입니다.
#    - 이를 위해 누군가 미리 정리해둔 '네이버 뉴스 요약 데이터셋'을 허깅페이스 허브에서 빌려옵니다.
#    - 데이터셋 ID: "daekeun-ml/naver-news-summarization-ko"
#
# 2. 데이터 구조 (Structure):
#    - DatasetDict: 데이터셋을 담고 있는 큰 가방입니다. 안에는 세 개의 주머니가 있습니다.
#      ① train (22,194개): 훈련용. 모델이 열심히 공부할 문제집.
#      ② validation (2,466개): 검증용. 학습 중간중간 실력 체크하는 모의고사.
#      ③ test (2,740개): 테스트용. 학습 다 끝나고 최종 평가하는 수능.
#
# 3. 컬럼(Features) 의미:
#    - date, category, press...: 기사 정보들. (학습엔 안 쓸 예정)
#    - document: **핵심 입력**. 뉴스 기사 본문.
#    - summary: **핵심 정답**. 사람이 요약해 둔 문장.
#    -> 나중에 이 'document'와 'summary'만 뽑아서 쓸 겁니다.
# =============================================================================

from datasets import load_dataset

# -----------------------------------------------------------------------------
# 1. 데이터셋 다운로드
# -----------------------------------------------------------------------------
# load_dataset("ID"):
# -> 허깅페이스 클라우드 서버에서 해당 ID의 데이터를 내 Colab으로 복사해옵니다.
# -> 인터넷 속도에 따라 몇 초에서 몇 분 걸릴 수 있습니다.
dataset = load_dataset("daekeun-ml/naver-news-summarization-ko")

# -----------------------------------------------------------------------------
# 2. 데이터셋 구성 확인
# -----------------------------------------------------------------------------
# 변수 이름(dataset)을 그냥 적으면, 그 안의 구조(메타데이터)를 보여줍니다.
dataset

# [출력 결과 해석]
# DatasetDict({
#     train: Dataset({
#         features: ['date', ... 'document', ... 'summary'], -> 컬럼 목록
#         num_rows: 22194                                    -> 데이터 개수 (약 2.2만 개)
#     })
#     ...
# })
# -> 데이터가 깨지지 않고 3등분(train, validation, test)되어 잘 들어왔다는 뜻입니다.
# -> 이제 다음 단계에서 이 데이터를 모델 입맛에 맞게 가공하면 됩니다.

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [ ]:
# =============================================================================
# [Step 4-2: 데이터 상세 확인 (Data Inspection)]
# =============================================================================
#
# 1. 실행 결과 분석 (Output Analysis):
#    - dataset['train'][0]을 실행하니 딕셔너리 형태의 데이터 하나가 나왔습니다.
#
#    {
#      'date': '2022-07-03...',   -> 날짜 (학습에 안 씀)
#      'category': 'economy',     -> 카테고리 (학습에 안 씀)
#      'press': 'YTN',            -> 언론사 (학습에 안 씀)
#      'title': '추경호...',      -> 제목 (경우에 따라 쓰지만, 여기선 본문 요약이 목적이라 패스)
#
#      'document': '앵커 정부가...'
#       -> [핵심 입력(Input/Feature)]: 모델이 읽어야 할 "뉴스 기사 원문"입니다.
#       -> 나중에 챗봇 프롬프트의 "User(질문자)" 역할에 들어갈 내용입니다.
#
#      'link': 'https://...',     -> 링크 (학습에 안 씀)
#
#      'summary': '올해 상반기...'
#       -> [핵심 정답(Output/Label)]: 모델이 뱉어내야 할 "요약문"입니다.
#       -> 나중에 챗봇 프롬프트의 "Assistant(답변자)" 역할에 들어갈 내용입니다.
#    }
#
# 2. 다음 단계 예고 (Next Step):
#    - Gemma 모델은 "채팅(대화)" 형태로 학습된 모델입니다.
#    - 그냥 이 데이터를 툭 던져주면 이해를 잘 못합니다.
#    - 그래서 이 데이터를 대화 형식으로 바꿔주는 "성형 수술(Preprocessing)"을 해야 합니다.
#      (예: User: "이 기사 요약해줘\n[document 내용]" -> Assistant: "[summary 내용]")
# =============================================================================
dataset['train'][0]

{'date': '2022-07-03 17:14:37',
 'category': 'economy',
 'press': 'YTN ',
 'title': '추경호 중기 수출지원 총력 무역금융 40조 확대',
 'document': '앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.',
 'link': 'https://n.news.naver.com/mne

### 챗봇 프롬프트 형식 변환

gemma-3-1b-it은 챗봇 형식의 프롬프트를 입력으로 받으므로 프롬프트 변환 함수를 정의하고 데이터세트에 `map()` 함수를 통해 변환 과정을 진행합니다.  

또한 `remove_columns` 인자를 통해 기존 데이터세트에 존재한 컬럼을 전부 제거합니다.

In [ ]:
# =============================================================================
# [Step 5: 데이터 포맷 변환 - "야, 너두 대화할 수 있어"]
# =============================================================================
#
# 1. 개념 (Concept):
#    - Gemma-it (Instruction Tuned) 모델은 "채팅"을 위해 태어난 모델입니다.
#    - 따라서 학습 데이터도 '질문(User)'과 '답변(Assistant)'이 주고받는 대화 형식이어야 학습 효율이 최고입니다.
#    - 기존: {'document': '뉴스내용', 'summary': '요약내용'}
#    - 변경: [{'role': 'user', 'content': '뉴스내용'}, {'role': 'assistant', 'content': '요약내용'}]
#
# 2. 코드 설명 (Line by Line):
# =============================================================================

# -----------------------------------------------------------------------------
# 1. 변환 함수 정의 (Rules)
# -----------------------------------------------------------------------------
def create_conversation(sample):
    # sample: 데이터 한 줄(row)이 들어옵니다.
    return {
        "messages": [
            # ① User(사용자)의 역할: 뉴스 본문(document)을 던져주는 역할
            {"role": "user", "content": sample["document"]},

            # ② Assistant(AI)의 역할: 그에 맞는 요약(summary)을 대답하는 역할
            {"role": "assistant", "content": sample["summary"]}
        ]
    }

# -----------------------------------------------------------------------------
# 2. 전체 데이터에 적용 (Mapping)
# -----------------------------------------------------------------------------
# dataset['train'].map(...): 훈련 데이터 전체를 돌면서 위 함수를 적용합니다.
train_data = dataset['train'].map(
    create_conversation,

    # remove_columns: **매우 중요**
    # -> 변환이 끝나면 기존의 'document', 'summary', 'date' 같은 잡다한 컬럼은 필요 없습니다.
    # -> 용량만 차지하고 헷갈리니까 싹 지워버립니다. (features 목록에 있는 거 전부 삭제)
    remove_columns=dataset['train'].features,

    batched=False # 한 번에 하나씩 정확하게 처리합니다.
)

# 테스트 데이터도 똑같이 처리해줍니다.
test_data = dataset['test'].map(
    create_conversation,
    remove_columns=dataset['test'].features,
    batched=False
)

# -----------------------------------------------------------------------------
# 3. 변환 결과 확인
# -----------------------------------------------------------------------------
train_data[0]

# [결과 해석]
# {'messages': [
#    {'content': '앵커 정부가...', 'role': 'user'},       -> 입력(Input) 준비 완료
#    {'content': '올해 상반기...', 'role': 'assistant'}   -> 정답(Label) 준비 완료
# ]}
# -> 이제 이 덩어리를 모델에게 던져주면,
#    "User가 이렇게 말했을 때, Assistant처럼 말하는 법을 배워!"라고 학습하게 됩니다.

{'messages': [{'content': '앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.',
   'role': 'user'},
  {'content': '올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, 정부가 하반기에 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 결정한 가운데, 특히 수출 중소

## Lora 옵션 세팅

Lora는 효율적인 파라미터 업데이트를 위해 각 projection 레이어층에 저차원 행렬 분해가 이루어진 Adapter 층을 추가하고 해당 Adapter 레이어만 학습을 진행합니다.

<center><img src="https://drive.google.com/uc?export=view&id=1aF59YFbuOUM32_ZComXLroBkhNV1bAav" width="600"/></center>

학습이 완료되면 Adapter 층의 분해 행렬을 행렬곱 연산하여 기존 project 레이어와 동일 형상으로 만든뒤 더하기 연산을 통해 가중치를 합칩니다.



허깅페이스에서 Lora 방식으로 미세조정을 하기 위해서 peft 모듈의 `LoraConfig` 객체를 통해 Lora 옵션을 설정해 줍니다.

- r(Rank)

    * LoRA에서 추가 학습 파라미터를 저장할 때 사용되는 저랭크(low-rank) 차원의 크기

    * (hidden_size × hidden_size) => (hidden_size × r) × (r × hidden_size)

    * r이 클수록 모델이 표현할 수 있는 변화(적응) 폭이 커지지만, 그만큼 추가 파라미터가 늘어나고 계산량이 많아짐

- lora_alpha

    * LoRA에서 추가로 학습되는 저랭크 가중치 행렬을 스케일링 하기위한 변수 LoRA출력 × (lora_alpha / r)

    * 모델이 학습 초기에 급격하게 파라미터 업데이트를 하지 않도록 조정하거나, 학습 후반부에 미세한 조정을 가능케 하는 등 학습 안정성과 성능에 영향을 줌

- target_modules

    * LoRA를 삽입할 내부 레이어 계층 이름을 입력

    * LoRA는 주로 어텐션 계층의 선형 레이어(projection)에 삽입하여 활용

    * "all-linear" 를 입력하여 모델의 모든 projection 레이어에 추가 가능


In [ ]:
# =============================================================================
# [Step 6: LoRA 설정 (Config) - "뇌의 어느 부위를 얼마나 고칠까?"]
# =============================================================================
#
# 1. 개념 (Concept):
#    - LoRA는 거대한 모델 전체를 수술하는 게 아니라, '어댑터'라는 작은 보조 장치만 붙여서 학습하는 기술입니다.
#    - 여기서 그 '보조 장치의 크기(r)'와 '부착 위치(target_modules)'를 정합니다.
#
# 2. 주요 변경점 (vs BERT):
#    - BERT: task_type="SEQ_CLS" (분류 문제)
#    - Gemma: task_type="CAUSAL_LM" (문장 생성 문제)
#    - Target Modules: BERT는 보통 query, value만 건드리지만, LLM은 q, k, v, o 전부 건드리는 게 성능이 좋습니다.
# =============================================================================

from peft import LoraConfig

lora_config = LoraConfig(
    # -------------------------------------------------------------------------
    # 1. 학습 용량 설정 (Rank)
    # -------------------------------------------------------------------------
    r=16,
    # -> LoRA 행렬의 '랭크(Rank)'입니다. 어댑터의 두께라고 생각하면 됩니다.
    # -> 숫자가 클수록: 더 똑똑해지지만, 메모리를 많이 먹고 느려집니다.
    # -> 숫자가 작을수록: 가볍고 빠르지만, 복잡한 내용을 배우기 힘듭니다.
    # -> 16은 LLM 파인튜닝에서 성능과 효율을 모두 잡는 '국룰(표준)' 값입니다.

    # -------------------------------------------------------------------------
    # 2. 영향력 조절 (Alpha)
    # -------------------------------------------------------------------------
    lora_alpha=8,
    # -> '새로 배운 지식(LoRA)'을 '원래 지식(Pre-trained)'에 얼마나 강하게 섞을지 결정합니다.
    # -> 공식: 최종 가중치 = 원래 가중치 + (LoRA 가중치 * alpha / r)
    # -> 여기선 8/16 = 0.5니까, 새로 배운 내용을 절반 정도 힘으로 반영하겠다는 뜻입니다.

    # -------------------------------------------------------------------------
    # 3. 과적합 방지 (Dropout)
    # -------------------------------------------------------------------------
    lora_dropout=0.05,
    # -> 학습할 때 뉴런의 5%를 랜덤하게 꺼버립니다.
    # -> "달달 외우지 말고, 응용력을 키워라"라는 의미입니다. (오버피팅 방지)

    # -------------------------------------------------------------------------
    # 4. 수술 부위 지정 (Target Modules) - 매우 중요!
    # -------------------------------------------------------------------------
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],
    # -> 아까 `print(model)` 했을 때 봤던 그 이름들입니다!
    # -> Attention 메커니즘의 핵심 부품 4가지(Query, Key, Value, Output)에 모두 어댑터를 붙입니다.
    # -> 이렇게 다 붙여야 한국어 요약 같은 복잡한 작업을 잘 배웁니다.
    # -> (참고: mlp 레이어는 건드리지 않아서 메모리를 아낍니다)

    # -------------------------------------------------------------------------
    # 5. 작업 유형 (Task Type) - BERT와 가장 큰 차이
    # -------------------------------------------------------------------------
    task_type="CAUSAL_LM"
    # -> Causal Language Modeling (인과적 언어 모델링)
    # -> "앞 단어를 보고 뒷 단어를 예측하는 모델" (GPT, Gemma 계열)
    # -> BERT 때는 분류 문제라 "SEQ_CLS"였지만, 지금은 생성 문제라 이걸 씁니다.
)

**LoRA task type (Low-Rank Adaptation)**

- task_type="SEQ_CLS"
    - 시퀀스 분류 (Sequence Classification)
    - 모델: BERT, RoBERTa 등

- task_type="SEQ_2_SEQ_LM"
    - 시퀀스-투-시퀀스 언어 모델링
    - 모델: T5, BART, mT5 등

- task_type="CAUSAL_LM"
    - 인과적(자기회귀) 언어 모델링
    - 모델: GPT, Llama, Gemma 등

https://huggingface.co/learn/smol-course/ko/unit3/2


## PEFT 학습 진행

AutoModelForCausalLM 처럼 교사 주도(supervised) 학습 방식을 사용하는 생성형 모델은 trl 모듈의 `SFTConfig(Supervised Fine-Tuning Config)` 를 통해 하이퍼파라미터를 설정하고 `SFTTrainer`로 쉽게 허깅페이스 프레임에 맞춰 데이터를 학습할 수 있습니다.

BERT 모델 예시에서 사용한 `TrainingArguments`, `Trainer`를 상속하여 확장된 객체이기 때문에 기본적으로 동일한 방식으로 사용하며, 기존 파라미터에 텍스트 생성 모델의 미세조정에 특화된 파라미터를 추가로 활용할 수 있습니다.  


https://huggingface.co/docs/trl/sft_trainer

**SFTConfig 하이퍼파라미터 설정**

TrainingArguments 과 동일한 인자 이름으로 모델 학습에 필요한 하이퍼파라미터를 설정합니다.

- TrainingArguments를 상속받아 확장
- LLM Fine-tuning 특화


In [ ]:
# =============================================================================
# [Step 7: 학습 하이퍼파라미터 설정 (SFTConfig)]
# =============================================================================
#
# 1. 개념 (Concept):
#    - SFTConfig는 Hugging Face의 TrainingArguments를 상속받은 설정집입니다.
#    - 컴퓨터 자원(GPU 메모리)은 한정되어 있고, 모델(Gemma)은 크기 때문에 "타협"이 필요합니다.
#    - 여기서 설정하는 값들은 대부분 "메모리가 터지지 않게 하면서 + 최대한 빠르게 + 성능 좋게" 학습하기 위한 전략입니다.
#
# 2. 핵심 전략 (Key Strategy):
#    ① 가상 배치 크기 (Gradient Accumulation):
#       - GPU가 작아서 한 번에 2문제(Batch=2)밖에 못 풉니다.
#       - 근데 2문제 풀고 정답 맞추면 공부가 잘 안 됩니다. (너무 자주 방향을 바꿈)
#       - 그래서 "2문제씩 4번 풀어서(총 8문제) 답을 모은 뒤에 한 번에 채점"합니다.
#       - 이게 바로 `gradient_accumulation_steps`입니다.
#
#    ② 혼합 정밀도 (fp16):
#       - 32비트(기본) 대신 16비트로 계산해서 속도를 2배 높이고 메모리를 절약합니다.
#
# 3. 코드 설명 (Line by Line):
# =============================================================================

from trl import SFTConfig

# 저장할 폴더 이름 설정
save_model = "gemma-text-to-sum4"

# SFTConfig: "선생님, 이렇게 가르쳐주세요" 지침서 작성
args = SFTConfig(
    # -------------------------------------------------------------------------
    # 1. 기본 설정 (Basics)
    # -------------------------------------------------------------------------
    output_dir=save_model,
    # -> 학습 중간중간 저장되는 체크포인트와 최종 모델이 저장될 폴더 경로입니다.

    num_train_epochs=1,
    # -> 데이터셋 전체를 '1바퀴'만 돌립니다.
    # -> 뉴스 요약 같은 작업은 금방 배우기 때문에, 많이 돌리면 오히려 멍청해질(Overfitting) 수 있습니다.

    eval_strategy="no",
    # -> "중간 평가는 하지 마." (시간 절약을 위해 생략)
    # -> 제대로 하려면 'steps'나 'epoch'으로 설정하고 검증 데이터(validation)를 줍니다.

    do_eval=False,
    # -> 위와 동일하게 평가 기능을 끕니다.

    # -------------------------------------------------------------------------
    # 2. 배치 처리 전략 (Batch Strategy) - 중요!
    # -------------------------------------------------------------------------
    per_device_train_batch_size=2,
    # -> "GPU 하나당 한 번에 2개의 데이터만 처리해."
    # -> LLM은 덩치가 커서 이 숫자를 높이면 바로 'OOM(Out Of Memory)' 에러가 납니다.
    # -> Colab 무료 버전(T4)에서는 보통 1~4 정도가 한계입니다.

    gradient_accumulation_steps=4,
    # -> "배치가 너무 작으니까, 4번 모았다가(Accumulate) 가중치를 업데이트해."
    # -> 실제 효과: 2(배치) x 4(누적) = 8개의 데이터를 한 번에 본 것과 같은 효과(Effective Batch Size)를 냅니다.
    # -> 학습 안정성을 높여주는 꿀팁입니다.

    # -------------------------------------------------------------------------
    # 3. 메모리 및 속도 최적화 (Optimization)
    # -------------------------------------------------------------------------
    gradient_checkpointing=False,
    # -> "중간 계산 결과(Checkpoint)를 저장하지 마."
    # -> True로 하면 메모리를 엄청 아낄 수 있지만 속도가 느려집니다.
    # -> 지금 쓰는 모델(1B)은 작아서 False(속도 우선)로 해도 괜찮습니다. (7B 이상은 True 필수)

    fp16=True,
    # -> "계산할 때 16비트(Half Precision)를 써."
    # -> 최신 GPU에서는 속도가 훨씬 빠르고 메모리도 적게 먹습니다. (성능 차이는 거의 없음)

    # -------------------------------------------------------------------------
    # 4. 학습 속도 조절 (Learning Rate)
    # -------------------------------------------------------------------------
    learning_rate=2e-4,
    # -> 학습률 0.0002
    # -> 일반적인 미세조정(1e-5)보다 좀 큽니다.
    # -> 이유: LoRA는 학습할 파라미터가 워낙 적어서, 좀 과감하게(크게) 학습시켜야 효과가 나옵니다.

    optim="adamw_torch_fused",
    # -> 최적화 알고리즘(Optimizer). "성적을 올리는 방법"입니다.
    # -> 'fused': GPU 하드웨어 가속을 써서 더 빠른 버전을 사용합니다.

    lr_scheduler_type="constant",
    # -> "학습률을 바꾸지 말고 처음부터 끝까지 0.0002로 쭉 밀고 나가."
    # -> 보통은 점점 줄이는(linear) 방식을 쓰지만, 1 epoch 짧은 학습이라 그냥 고정했습니다.

    warmup_ratio=0.03,
    # -> "맨 처음 3% 구간은 살살(0부터 2e-4까지 서서히 올림) 해줘."
    # -> 시작하자마자 냅다 달리면 모델이 놀라서(발산해서) 학습이 망가질 수 있습니다. 준비운동 구간입니다.

    # -------------------------------------------------------------------------
    # 5. 안정성 및 기타 (Stability & Misc)
    # -------------------------------------------------------------------------
    max_grad_norm=0.3,
    # -> "기울기(Gradient)가 0.3을 넘으면 강제로 잘라버려(Clipping)."
    # -> 학습이 튀는 것(Gradient Explosion)을 막는 안전장치입니다.

    weight_decay=2e-4,
    # -> "가중치가 너무 커지지 않게 규제(Regularization)해줘." (과적합 방지)

    # -------------------------------------------------------------------------
    # 6. 기록 및 저장 (Logging & Saving)
    # -------------------------------------------------------------------------
    logging_steps=100,
    # -> "100 스텝 지날 때마다 터미널에 Loss(오차) 점수 찍어줘." (잘 되고 있나 확인용)

    save_strategy="epoch",
    # -> "저장은 1 Epoch 다 돌면 그때 한 번 해."

    save_total_limit=2,
    # -> "저장공간 부족하니까 모델 파일은 최근 2개까지만 남기고 옛날 건 지워."

    load_best_model_at_end=False,
    # -> "끝나고 나서 제일 잘했던 모델 다시 불러올 필요 없어." (어차피 Epoch 1번이라 마지막이 최선임)

    push_to_hub=False,
    report_to="none"
    # -> "외부 사이트(허깅페이스 허브, 완드비 등)에 올리지 마."
)

### SFTTrainer 모델 학습

`args`에는 `SFTConfig`을, `peft_config`에는 `LoraConfig`를 입력하여 모델에 Lora 미세조정을 시작합니다. 추가로 토큰의 클래스 정보를 위해 `processing_class` 인자에 토크나이져를 입력합니다.

In [ ]:
# =============================================================================
# [Step 8: SFTTrainer 초기화 - "모든 준비는 끝났다"]
# =============================================================================
#
# 1. 개념 (Concept):
#    - SFTTrainer (Supervised Fine-tuning Trainer):
#      -> 지도 학습(정답이 있는 공부)을 위해 특화된 관리자입니다.
#      -> 우리가 따로 LoRA를 모델에 적용하는 코드를 짜지 않아도,
#         'peft_config'만 넣어주면 내부적으로 알아서 모델에 LoRA를 장착합니다.
#
# 2. 로그 해석 (Output Analysis):
#    - Tokenizing train dataset: 100%
#      -> 선생님이 교재(데이터셋)를 받아서 모델이 읽을 수 있는 숫자(Token)로 변환했습니다.
#    - Truncating train dataset: 100%
#      -> 너무 긴 문장은 잘라내서, 모델이 한 번에 소화할 수 있는 길이로 맞췄습니다.
#    - 결론: 학습(train) 버튼만 누르면 바로 시작할 수 있는 상태입니다.
#
# 3. 코드 설명 (Line by Line):
# =============================================================================

from trl import SFTTrainer

# Trainer 객체 생성 (이 변수가 학습의 모든 것을 관장합니다)
trainer = SFTTrainer(
    # -------------------------------------------------------------------------
    # 1. 학생 (Model)
    # -------------------------------------------------------------------------
    model=model,
    # -> 아까 준비한 4비트 양자화된 Gemma 모델입니다.

    # -------------------------------------------------------------------------
    # 2. 진도표 및 규칙 (Arguments)
    # -------------------------------------------------------------------------
    args=args,
    # -> SFTConfig로 설정한 학습 규칙(배치 사이즈, 학습률 등)입니다.

    # -------------------------------------------------------------------------
    # 3. 교과서 (Dataset)
    # -------------------------------------------------------------------------
    train_dataset=train_data,
    # -> 채팅 포맷으로 변환된 훈련 데이터셋입니다.

    # -------------------------------------------------------------------------
    # 4. 공부법 (LoRA Config) - 핵심!
    # -------------------------------------------------------------------------
    peft_config=lora_config,
    # -> 이 설정이 들어가면 Trainer가 알아서 "아, 전체 학습 말고 LoRA만 시켜야지"라고 인식합니다.
    # -> 모델에 LoRA 어댑터를 자동으로 부착해줍니다.

    # -------------------------------------------------------------------------
    # 5. 번역기 (Tokenizer)
    # -------------------------------------------------------------------------
    processing_class=tokenizer
    # -> 데이터를 숫자로 바꾸고, 정답을 만들 때 사용할 토크나이저입니다.
    # -> (참고: 예전엔 'tokenizer'라는 이름이었는데, 최근엔 이미지 처리까지 포함해서 'processing_class'로 이름이 바뀌는 추세입니다)
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
# =============================================================================
# [Step 9: 학습 완료 및 모델 저장 - "드디어 AI가 깨달음을 얻었다"]
# =============================================================================
#
# 1. 학습 로그 해석 (Log Analysis):
#    - [2775/2775 1:39:25, Epoch 1/1]
#      -> 총 2,775번의 문제 풀이(Step)를 완료했습니다.
#      -> 시간은 약 1시간 40분이 걸렸네요.
#
#    - Loss (손실값) 변화:
#      Step 100: 2.6084 -> Step 2700: 2.0847
#      -> 숫자가 줄어들고 있다는 건, 모델이 정답(요약문)을 맞추는 확률이 높아졌다는 뜻입니다.
#      -> 급격하게 떨어지진 않았지만, 꾸준히 우하향 곡선을 그리는 "아주 건강한 학습 상태"입니다.
#
# 2. 경고 메시지 (The tokenizer has new PAD/BOS/EOS tokens...):
#    - "토크나이저랑 모델 설정이랑 특수 문자(문장 끝 등) 번호가 살짝 달라서 내가 맞춰줬어."
#    - 라이브러리가 자동으로 해결해준 것이므로 **무시하셔도 됩니다.**
#
# 3. 코드 설명 (Line by Line):
# =============================================================================

# -----------------------------------------------------------------------------
# 1. 학습 실행 (Train)
# -----------------------------------------------------------------------------
# trainer.train():
# -> 모델이 "문제 풀기(Forward) -> 채점(Loss) -> 오답 노트 작성(Backward) -> 뇌 수정(Update)"
#    이 과정을 2,775번 반복했습니다.
# -> 이 한 줄이 실행되는 동안 GPU는 엄청나게 뜨거워졌을 겁니다.
trainer.train()

# -----------------------------------------------------------------------------
# 2. 모델 저장 (Save) - 중요!
# -----------------------------------------------------------------------------
# trainer.save_model():
# -> 학습된 결과물을 'gemma-text-to-sum4' 폴더에 저장합니다.
#
# [저장되는 파일의 정체]
# -> 여기서 저장되는 건 수 기가바이트짜리 Gemma 모델 전체가 아닙니다.
# -> 우리가 아까 설정한 **LoRA 어댑터 파일(약 20~30MB)**만 저장됩니다.
# -> 파일 목록:
#    - adapter_config.json: "이 어댑터는 r=16이고 alpha=8이야" 같은 설명서
#    - adapter_model.safetensors: 실제 학습된 가중치 파일 (핵심)
#    - tokenizer 설정 파일들: 나중에 똑같이 번역하기 위해 필요
trainer.save_model()

# =============================================================================
# [다음 단계 예고]
# 이제 학습된 '어댑터'만 저장된 상태입니다.
# 이걸 실제로 써먹으려면
# 1. 원본 모델(Gemma)을 다시 불러오고,
# 2. 방금 저장한 어댑터(LoRA)를 그 위에 '합체(Merge)'시켜야 합니다.
# 다음 코드에서 이 작업을 수행합니다.
# =============================================================================

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss
100,2.608400
200,2.373900
300,2.323900
400,2.265700
500,2.285800
600,2.241600
700,2.194700
800,2.233500
900,2.162200
1000,2.156900


## Lora 모델 통합

Lora 미세조정이 완료되면 결과적으로 추가된 Lora 계층 (저차원 분해 행렬 Adapter)의 가중치를 얻게됩니다.

최종 적으로 해당 가중치를 기존 모델에 통합하여 배포 되어야만 미세조정의 결과가 적용된 모델로 활용이 가능합니다.  

In [ ]:
# =============================================================================
# [Step 10: 결과물 압축 및 GPU 메모리 대청소]
# =============================================================================
#
# 1. 압축 (Archive):
#    - 학습된 LoRA 어댑터 파일들은 폴더에 흩어져 있습니다.
#    - 이걸 'zip' 파일 하나로 묶어서 나중에 다운로드하거나 구글 드라이브로 옮기기 쉽게 만듭니다.
#
# 2. 메모리 청소 (Garbage Collection):
#    - **가장 중요한 단계입니다.**
#    - 다음 단계에서 우리는 "원본 모델"을 다시 불러와야 합니다.
#    - 근데 방금 학습에 쓴 모델(model)과 선생님(trainer)이 GPU 자리를 차지하고 있습니다.
#    - del 명령어로 파이썬 변수를 지우고, empty_cache()로 GPU에 남은 찌꺼기 데이터를 싹 비워줍니다.
# =============================================================================

import shutil
from google.colab import files

# -----------------------------------------------------------------------------
# 1. 압축 파일 생성
# -----------------------------------------------------------------------------
# shutil.make_archive(저장할이름, 포맷, 대상폴더)
# -> 'gemma-text-to-sum4' 폴더를 'gemma-text-to-sum4.zip'으로 압축합니다.
# -> 왼쪽 파일 탐색기(폴더 아이콘)를 새로고침 해보면 zip 파일이 생긴 걸 볼 수 있습니다.
shutil.make_archive(save_model, 'zip', save_model)

# (참고: 다운로드하려면 files.download('gemma-text-to-sum4.zip')을 실행하면 됩니다)

'/content/gemma-text-to-sum4.zip'

> 약 26MB 크기의 Lora 체크포인트가 만들어집니다.

In [ ]:
# -----------------------------------------------------------------------------
# 2. 메모리 초기화 (청소)
# -----------------------------------------------------------------------------
# del: 파이썬 프로그램 내에서 변수 연결을 끊어버립니다. (메모리에서 놓아줌)
del model
del trainer

# torch.cuda.empty_cache():
# -> 파이토치가 관리하던 GPU 캐시 메모리를 강제로 비웁니다.
# -> 이걸 안 하면 작업 관리자에는 안 보이는데 GPU 메모리가 꽉 차 있는 '유령 메모리' 현상이 생깁니다.
torch.cuda.empty_cache()

# -> 이제 GPU가 깨끗해졌으므로, 다음 단계(Merge)를 안전하게 진행할 수 있습니다.

### Lora 가중치 합치기

기존 모델에 Lora 학습 결과를 통합하는 방법은 먼저 Lora의 저차원 분해 행렬을 행렬곱 하여 Lora가 추가되었던 projection 레이어와 동일한 형상으로 맞춰준다음 더하기 연산을 진행합니다.

Lora 가중치 통합을 위해 peft 모듈의 `PeftModel` 객체에서 기본 모델과 저장 경로를 함께 입력하여 불러오고 `merge_and_unload()` 함수를 호출합니다.



In [ ]:
# =============================================================================
# [Step 11: 모델 병합 (Merge) - "따로 놀던 두 뇌를 하나로 합체"]
# =============================================================================
#
# 1. 개념 (Concept):
#    - 지금 상태: Base Model(원본) + Adapter(학습된 추가 파일)가 따로 존재함.
#    - 목표: Base Model의 가중치(W)에 Adapter의 가중치(Delta W)를 수학적으로 더해버림.
#    - 결과: LoRA 파일 없이도 독자적으로 실행 가능한 하나의 완성된 모델 탄생.
#
# 2. 경고 메시지 해석 (Warning Analysis):
#    - "Merge lora module to 8-bit linear may get different generations due to rounding errors."
#    - 해석: "원본 모델을 8비트(정수)로 불러와서 합치고 있는데, LoRA는 소수점(실수)이라서
#             더하는 과정에서 반올림 오차(Rounding Error)가 생길 수 있어."
#    - 영향: 학습할 때 성능과 100% 똑같지는 않을 수 있지만, 대게 미미한 차이라 넘어갑니다.
#            (완벽하게 하려면 원본을 16비트로 로드해서 합쳐야 하지만 메모리가 많이 듭니다.)
#
# 3. 코드 설명 (Line by Line):
# =============================================================================

from peft import PeftModel

# -----------------------------------------------------------------------------
# 1. 원본 모델 다시 불러오기 (이번엔 8비트로)
# -----------------------------------------------------------------------------
# 4비트(학습용)가 아니라 8비트 설정을 씁니다.
# 병합할 때는 4비트보다 8비트나 16비트가 계산 오차가 적기 때문입니다.
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# 텅 빈 원본 Gemma 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config
).eval() # 학습 모드 끄기(eval)

# -----------------------------------------------------------------------------
# 2. 합체 준비 (Mounting)
# -----------------------------------------------------------------------------
# PeftModel.from_pretrained:
# -> 원본 모델(model) 위에, 아까 저장해둔 LoRA 어댑터(save_model 폴더)를 끼웁니다.
# -> 아직 합쳐진 건 아니고, "장착"만 된 상태입니다.
peft_model = PeftModel.from_pretrained(model, save_model)

# -----------------------------------------------------------------------------
# 3. 물리적 결합 (Merge & Unload) - 핵심!
# -----------------------------------------------------------------------------
# merge_and_unload():
# -> "LoRA 가중치를 원본 가중치에 더해버리고(Merge), LoRA 껍데기는 버려라(Unload)."
# -> 이 함수가 실행되고 나면 'merged_model'은 더 이상 PEFT 모델이 아니라 그냥 일반 모델이 됩니다.
merged_model = peft_model.merge_and_unload()

# -----------------------------------------------------------------------------
# 4. 최종 모델 저장 (Serialization)
# -----------------------------------------------------------------------------
# merged_model.save_pretrained(...):
# -> 합쳐진 모델을 'merged_model'이라는 새 폴더에 저장합니다.
merged_model.save_pretrained(
    "merged_model",             # 저장할 폴더 이름
    safe_serialization=True,    # .bin 대신 .safetensors 포맷 사용 (보안상 안전하고 로딩 빠름)
    max_shard_size="2GB"        # "파일 하나가 2GB 넘으면 쪼개서 저장해." (다운로드 편의성)
)

# -----------------------------------------------------------------------------
# 5. 토크나이저 저장 (필수 세트)
# -----------------------------------------------------------------------------
# 모델만 저장하면 안 되고, 번역기(Tokenizer)도 세트로 같이 저장해야 합니다.
processor = AutoTokenizer.from_pretrained(save_model)
processor.save_pretrained("merged_model")

# =============================================================================
# [출력 결과 확인]
# ('merged_model/tokenizer_config.json', ...)
# -> 파일들이 정상적으로 생성되었습니다.
# -> 이제 'merged_model' 폴더 안에는 누구나 바로 사용할 수 있는
#    "나만의 뉴스 요약 전문 Gemma 모델"이 들어있습니다.
# =============================================================================

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:110: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


('merged_model/tokenizer_config.json',
 'merged_model/special_tokens_map.json',
 'merged_model/chat_template.jinja',
 'merged_model/tokenizer.model',
 'merged_model/added_tokens.json',
 'merged_model/tokenizer.json')

| 항목 | pickle (.bin) | safetensors (.safetensors) |
|:---:|:---:|:---:|
| **안전성** | ❌ 위험 | ✅ 안전 |
| **속도** | 빠름 | 더 빠름 ✅ |
| **크기** | 보통 | 작음 ✅ |
| **HuggingFace 지원** | 옛날 방식 | 최신 권장 ✅ |
| **코드 실행** | 가능 (위험) | 불가능 ✅ |
| **권장 여부** | ❌ | ✅ |

In [ ]:
# =============================================================================
# [Step 12: 최종 결과물 압축 - "이제 집에 가져가자"]
# =============================================================================
#
# 1. 개념 (Concept):
#    - 완성된 모델을 배포(Deployment)하거나 저장하기 위해 압축하는 과정입니다.
#    - Gemma-1B 모델(8bit)이라 압축 파일 용량은 대략 **1GB ~ 2GB** 사이가 됩니다.
#
# 2. 코드 설명 (Line by Line):
# =============================================================================

import shutil
from google.colab import files

# -----------------------------------------------------------------------------
# 폴더 압축하기
# -----------------------------------------------------------------------------
# shutil.make_archive(
#     base_name='merged_model',  -> 생성될 파일 이름 (확장자 제외)
#     format='zip',              -> 압축 방식
#     root_dir='merged_model'    -> 압축할 대상 폴더
# )
# -> 결과: '/content/merged_model.zip' 파일이 생성됩니다.
shutil.make_archive('merged_model', 'zip', 'merged_model')

# [참고: 다운로드 방법]
# 코랩에서 로컬 컴퓨터로 다운로드하려면 아래 코드를 추가로 실행해야 합니다.
# (단, 1GB가 넘으면 브라우저 다운로드는 자주 끊기니 구글 드라이브 저장을 추천합니다)
# files.download('merged_model.zip')

'/content/merged_model.zip'

> 약 1GB 크기의 통합 모델이 저장됩니다

In [ ]:
# =============================================================================
# [Step 13: 어댑터 로드 및 테스트 준비]
# "합치지 않았는데 어떻게 실행되나요?" -> 라이브러리의 마법입니다.
# =============================================================================
#
# 1. 작동 원리 (Mechanism):
#    - 우리가 `save_model` 변수에 담긴 경로("./gemma-text-to-sum4")를 호출합니다.
#    - 이 폴더 안에는 'adapter_config.json'이라는 설정 파일이 있습니다.
#    - 라이브러리가 이 파일을 딱 열어보고:
#      "아하! 이 어댑터의 주인(Base Model)은 'google/gemma-3-1b-it'이구나!" 라고 파악합니다.
#    - 그래서 [원본 모델 다운로드/로드] -> [어댑터 장착] 과정을 순식간에 자동으로 처리합니다.
#
# 2. 코드 설명 (Line by Line):
# =============================================================================

import torch
from transformers import pipeline

# -----------------------------------------------------------------------------
# 1. 모델 불러오기 (AutoModelForCausalLM)
# -----------------------------------------------------------------------------
# 여기서 'save_model'은 20MB짜리 LoRA 어댑터 폴더입니다.
# 1GB짜리 'merged_model'을 쓰지 않아도 똑같이 동작합니다.
my_model = AutoModelForCausalLM.from_pretrained(
    save_model,
    # -> 경로에 어댑터 폴더를 넣었지만, 실제로는 원본 Gemma 모델까지 같이 불러옵니다.

    device_map='auto'
    # -> [메모리 관리의 핵심]
    # -> "GPU 메모리가 넉넉하면 GPU에 올리고, 부족하면 CPU나 RAM으로 알아서 분산시켜라."
    # -> 이 옵션 덕분에 Colab에서 메모리 부족(OOM) 없이 안전하게 로드됩니다.
).eval()
# .eval():
# -> [평가 모드 전환]
# -> 학습할 때 썼던 'Dropout(랜덤하게 뉴런 끄기)' 기능을 비활성화합니다.
# -> 이걸 안 하면 같은 질문을 해도 매번 대답이 미세하게 달라질 수 있습니다.

# -----------------------------------------------------------------------------
# 2. 토크나이저 불러오기 (AutoTokenizer)
# -----------------------------------------------------------------------------
# 모델이 학습할 때 사용했던 번역기(Tokenizer) 설정을 그대로 가져옵니다.
# (Gemma의 기본 토크나이저에, 학습 과정에서 추가된 특수 토큰 정보 등이 포함될 수 있음)
tokenizer = AutoTokenizer.from_pretrained(save_model)

# -----------------------------------------------------------------------------
# [요약]
# 이 코드를 실행하면,
# 내 눈앞에는 'my_model'이라는 변수 하나만 보이지만
# 실제로는 [Gemma 원본] + [LoRA 어댑터]가 메모리 상에서 '가상으로 합체'된 상태입니다.
# =============================================================================

## 모델 평가

학습된 Lora 가중치를 활용하여 뉴스 요약이 잘 진행되는지 확인해 봅니다.

In [ ]:
# =============================================================================
# [Step 14: 프롬프트 엔지니어링 - "질문지 만들기"]
# =============================================================================
#
# 1. 개념 (Concept):
#    - 파이프라인(Pipeline): "전처리 -> 모델 추론 -> 후처리" 과정을 한 방에 처리해주는 도구입니다.
#    - 프롬프트 구성: 모델에게 그냥 뉴스 기사만 던져주면 "이걸 뭐 어쩌라고?" 하고 당황합니다.
#      그래서 "이건 유저가 말한 거고, 이제부터 네가 대답할 차례야"라는 형식을 만들어줘야 합니다.
#
# 2. 출력 결과 해석 (Prompt Analysis):
#    <bos>                       -> 문장 시작 (Beginning of Sentence)
#    <start_of_turn>user\n       -> "사용자 턴 시작!"
#    산업부·조달청...             -> (뉴스 기사 본문 내용)
#    <end_of_turn>\n             -> "사용자 말 끝!"
#    <start_of_turn>model\n      -> "자, 이제 모델(AI) 너의 턴이야. 여기서부터 말해!"
#
#    -> 이 마지막 <start_of_turn>model 태그가 있기에 모델이 바로 요약을 시작하는 겁니다.
#
# 3. 코드 설명 (Line by Line):
# =============================================================================

from transformers import pipeline

# -----------------------------------------------------------------------------
# 1. 파이프라인 생성 (만능 도구)
# -----------------------------------------------------------------------------
pipe = pipeline(
    "text-generation",  # "글짓기 도구를 꺼내라"
    model=my_model,     # 아까 로드한(LoRA 합체된) 모델 사용
    tokenizer=tokenizer # 짝꿍 토크나이저 사용
)

# -----------------------------------------------------------------------------
# 2. 테스트 데이터 하나 뽑기
# -----------------------------------------------------------------------------
# test_data[100]: 테스트 데이터셋의 100번째 뉴스를 가져옵니다.
# 이 데이터 안에는 'messages' 리스트가 있고, 그 안에 [질문, 정답]이 다 들어있습니다.
test_sample = test_data[100]

# -----------------------------------------------------------------------------
# 3. 프롬프트 조립 (핵심!)
# -----------------------------------------------------------------------------
prompt = pipe.tokenizer.apply_chat_template(
    test_sample["messages"][:1],
    # -> 중요!: [:1]을 쓴 이유
    #    데이터에는 [0: 질문(본문), 1: 정답(요약)]이 들어있습니다.
    #    모델에게 정답까지 보여주면 안 되니까, 0번(질문)만 잘라서 입력으로 줍니다.

    tokenize=False,
    # -> "숫자로 바꾸지 말고, 눈으로 볼 수 있게 글자(String)로 줘." (확인용)

    add_generation_prompt=True
    # -> "마지막에 '<start_of_turn>model' 태그를 붙여줘."
    # -> 이게 없으면 모델이 언제 말해야 할지 모릅니다.
)

# 조립된 프롬프트 확인
prompt

Device set to use cuda:0


'<bos><start_of_turn>user\n산업부·조달청 정책설명회 정부가 중견기업을 우리 산업의 새로운 성장 엔진으로 육성하기 위해 이들이 생산하는 제품을 공공조달을 통해 적극적으로 활용·육성하기로 했다. 중견기업은 우리나라 전체 공공조달 계약 규모 184조원 중 26조3000억원을 담당하고 있다. 산업통상자원부는 조달청과 함께 1일 서울 마포구 한국중견기업연합회에서 중견기업을 대상으로 공공조달 정책설명회를 개최하고 중견기업 제품의 공공조달 강화방안을 논의했다고 밝혔다. 이날 설명회는 중견기업 지원에 대한 정부인식 제고를 위해 마련된 가운데 조달청은 새 정부의 공공조달 정책방향을 설명했고 참석자들은 중견기업의 공공조달 관련 애로 및 건의 사항을 전달했다. 이종욱 조달청장은 “중견기업은 공공조달 시장 전체 기업 수의 0.7% 3487개 에 불과하지만 우리나라 전체 공공조달 계약 규모 184조원 중 26조3000억원을 담당한다”며 “중견기업이 공공조달 시장에 보다 활발하게 참여하고 국내를 넘어 해외조달시장으로 뻗어나갈 수 있도록 다각적인 지원방안을 강구하겠다”고 말했다. 장영진 사진 산업부 제1차관은 “우리 경제 역동성·활력을 제고하기 위해서는 기업 성장사다리의 핵심 연결고리인 중견기업에 대한 정부 지원체계 강화가 필수적”이라며 “이번 설명회를 시작으로 향후 중견기업계의 주요 애로사항과 관련된 부처를 대상으로 지속적인 소통을 하겠다”고 말했다.<end_of_turn>\n<start_of_turn>model\n'

In [ ]:
# =============================================================================
# [Step 15: 최종 생성 및 결과 확인 - "AI의 백일장"]
# =============================================================================
#
# 1. 개념 (Concept):
#    - 생성 파라미터(Generation Config): AI가 글을 쓸 때의 '성격'을 설정합니다.
#      -> 창의적으로 쓸지(Temperature 높게), 진지하게 쓸지(Temperature 낮게).
#      -> 언제 멈출지(EOS Token), 같은 말을 반복하면 혼낼지(No Repeat Ngram) 등.
#
# 2. 결과 분석 (Output Analysis):
#    - 생성된 문장: "중중기업과 조약청이... 공중조약 계약을 적극 활용하고..."
#    - 평가:
#      ① 성공 요소: 입력된 뉴스 기사의 핵심 키워드(중견기업, 조달청, 공공조달)를 인지하고 있습니다.
#      ② 부족한 점: 단어를 살짝 틀리게 말함 (조달청 -> 조약청).
#    - 원인:
#      1. 모델이 작음(1B): 사람으로 치면 초등학생 수준이라 어휘력이 부족함.
#      2. 학습 부족(1 Epoch): 교과서를 한 번만 훑어봐서 아직 헷갈림.
#      3. 양자화 손실(4bit): 뇌 용량을 줄이다 보니 디테일을 놓침.
#    - 해결책: 학습을 더 오래 시키거나(3 Epochs 이상), 더 큰 모델(7B)을 쓰면 완벽해집니다.
#
# 3. 코드 설명 (Line by Line):
# =============================================================================

# -----------------------------------------------------------------------------
# 1. 멈춤 신호 설정 (Stop Tokens)
# -----------------------------------------------------------------------------
# stop_token_ids:
# -> 모델이 말을 끝내야 할 때 사용하는 신호들입니다.
# -> eos_token_id: 문장의 끝 (.)
# -> <end_of_turn>: Gemma 모델이 대화를 마칠 때 쓰는 전용 신호
stop_token_ids = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<end_of_turn>")
]

# -----------------------------------------------------------------------------
# 2. 텍스트 생성 (Generation)
# -----------------------------------------------------------------------------
outputs = pipe(
    prompt,                 # 아까 만든 질문지(<start_of_turn>user...<start_of_turn>model)

    # --- 길이 조절 ---
    max_new_tokens=256,     # "최대 256단어까지만 써." (너무 길어지면 자름)

    # --- 창의성 조절 (Sampling) ---
    do_sample=True,         # "확률 높은 단어만 무조건 뽑지 말고, 가끔은 다른 단어도 뽑아봐." (다양성)

    temperature=0.5,        # "창의성 점수 0.5점."
                            # -> 0에 가까우면 기계적이고 정확하게, 1에 가까우면 아무말 대잔치(소설).
                            # -> 요약은 사실 전달이 중요하므로 0.5 정도로 낮게 잡습니다.

    top_k=50,               # "다음 단어 후보를 상위 50개 중에서만 골라." (이상한 단어 방지)
    top_p=0.9,              # "확률 합계가 90% 되는 애들 안에서만 골라." (핵심 단어 위주)

    # --- 제어 장치 ---
    eos_token_id=stop_token_ids, # "이 신호 나오면 바로 멈춰."

    disable_compile=False,       # 최적화 컴파일 끄기/켜기 (기본값)

    no_repeat_ngram_size=2       # "같은 단어 쌍(2개 연속)을 반복하면 벌점."
                                 # -> "해요해요해요" 같은 고장 난 레코드 현상 방지.
)

# -----------------------------------------------------------------------------
# 3. 최종 출력
# -----------------------------------------------------------------------------
# outputs 변수에는 입력 프롬프트와 생성된 문장이 합쳐져서 들어있습니다.
outputs

[{'generated_text': '<bos><start_of_turn>user\n산업부·조달청 정책설명회 정부가 중견기업을 우리 산업의 새로운 성장 엔진으로 육성하기 위해 이들이 생산하는 제품을 공공조달을 통해 적극적으로 활용·육성하기로 했다. 중견기업은 우리나라 전체 공공조달 계약 규모 184조원 중 26조3000억원을 담당하고 있다. 산업통상자원부는 조달청과 함께 1일 서울 마포구 한국중견기업연합회에서 중견기업을 대상으로 공공조달 정책설명회를 개최하고 중견기업 제품의 공공조달 강화방안을 논의했다고 밝혔다. 이날 설명회는 중견기업 지원에 대한 정부인식 제고를 위해 마련된 가운데 조달청은 새 정부의 공공조달 정책방향을 설명했고 참석자들은 중견기업의 공공조달 관련 애로 및 건의 사항을 전달했다. 이종욱 조달청장은 “중견기업은 공공조달 시장 전체 기업 수의 0.7% 3487개 에 불과하지만 우리나라 전체 공공조달 계약 규모 184조원 중 26조3000억원을 담당한다”며 “중견기업이 공공조달 시장에 보다 활발하게 참여하고 국내를 넘어 해외조달시장으로 뻗어나갈 수 있도록 다각적인 지원방안을 강구하겠다”고 말했다. 장영진 사진 산업부 제1차관은 “우리 경제 역동성·활력을 제고하기 위해서는 기업 성장사다리의 핵심 연결고리인 중견기업에 대한 정부 지원체계 강화가 필수적”이라며 “이번 설명회를 시작으로 향후 중견기업계의 주요 애로사항과 관련된 부처를 대상으로 지속적인 소통을 하겠다”고 말했다.<end_of_turn>\n<start_of_turn>model\n중중기업과 조약청이 중중개기업 기업의 생산 제품에 공중조약 계약을 적극 활용하고 육성을 위해 중소기업 4080개 중 중대기업 중 주관업체 885개에 대해 중기중대 기업을 지원하기 위한 정책을 발표했다는 가운데 중주관 5703개와 중장기 기업 604개 등 중공업을 위한 공조업계와 협력 방안에 중점했다 고 말했다고 밝혔다 가운데 정부는 조약을 통해 중지공공 조업 중 정부와 조화된 공통의 목표를 달성할 수 있다 고 강조했다

In [ ]:
# =============================================================================
# [Troubleshooting: 결과가 왜 'o' 한 글자만 나왔을까?]
# =============================================================================
#
# 1. 코드 분석:
#    outputs[0]['generated_text']   -> 전체 텍스트 (프롬프트 + 답변)
#    .strip('<start_of_turn>model') -> 문제의 구간!
#       -> strip은 '문자열 자르기'가 아니라 '양쪽 끝에서 해당 문자들을 제거'하는 함수입니다.
#       -> 의도와 다르게 작동했을 가능성이 큽니다.
#    [1] -> "그 결과물에서 인덱스 1번(두 번째 글자)만 가져와."
#       -> 그래서 뜬금없는 알파벳 하나('o')가 나온 것입니다.
#
# 2. 해결 방법 (Correct Way):
#    - strip 대신 split을 써야 합니다.
#    - "모델 발언 시작 부분(<start_of_turn>model)"을 기준으로 문장을 반으로 쪼개고,
#    - 뒷부분(실제 답변)만 가져와야 합니다.
# =============================================================================

# [수정된 코드]
# 1. 전체 텍스트 가져오기
full_text = outputs[0]['generated_text']

# 2. 모델의 답변 부분만 발라내기 (split 사용)
# <start_of_turn>model 이라는 신호탄을 기준으로 문장을 쪼갭니다.
# [-1]은 쪼개진 것 중 '맨 마지막 덩어리(답변)'를 의미합니다.
final_answer = full_text.split('<start_of_turn>model')[-1].strip()

print(final_answer)

# [출력 예시]
# "중견기업과 조달청이... (제대로 된 요약문이 나옵니다)"

'o'

In [ ]:
# =============================================================================
# [전체 시나리오: Google Gemma-3 모델을 '한국어 뉴스 요약 전문가'로 만들기]
# =============================================================================
#
# [Phase 1: 준비 단계]
#   Step 1: 환경 설정 및 라이브러리 설치
#   Step 2: 원본 모델(Gemma)을 4비트로 압축해서 불러오기 (QLoRA 준비)
#   Step 3: 학습 전, 멍청한 상태에서 한번 테스트해보기 (Pre-Inference)
#
# [Phase 2: 학습 단계 (Training)]
#   Step 4: 교과서(뉴스 데이터) 다운로드 및 확인
#   Step 5: 데이터를 채팅 포맷(User/Assistant)으로 변환 (전처리)
#   Step 6: LoRA 설정 (뇌의 어느 부위를 수술할지 결정)
#   Step 7: 학습 규칙 설정 (공부 스케줄 짜기 - SFTConfig)
#   Step 8: 선생님(Trainer) 배정 및 학습 시작
#   Step 9: 학습 종료 및 어댑터(LoRA) 저장
#
# [Phase 3: 배포 및 최종 확인 (Deployment)]
#   Step 10: 메모리 청소 (중요!)
#   Step 11: 원본 모델 + 학습된 어댑터 합체 (Merge)
#   Step 12: 합체된 모델 저장 및 압축
#   Step 13: 최종 모델 로드 (테스트용)
#   Step 14: 질문지(프롬프트) 만들기
#   Step 15: 최종 시험(추론) 및 결과 출력
# =============================================================================

# -----------------------------------------------------------------------------
# [Step 1] 라이브러리 설치 & 로그인
# -----------------------------------------------------------------------------
# !pip install ... (이미 설치했다고 가정)
from huggingface_hub import notebook_login
# notebook_login() # 토큰 입력 필요

import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, pipeline
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset
import shutil

# -----------------------------------------------------------------------------
# [Step 2] 모델 로드 & 양자화 (QLoRA)
# -----------------------------------------------------------------------------
# 구글의 Gemma-3 (10억 파라미터) 모델 ID
model_id = "google/gemma-3-1b-it"

# 4비트 양자화 설정 (메모리 절약을 위해 모델을 1/4 크기로 압축)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16, # 계산은 16비트로 정교하게
    bnb_4bit_quant_type="nf4",            # AI 가중치에 최적화된 포맷
    bnb_4bit_use_double_quant=True
)

# 모델 불러오기 (압축 적용)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    attn_implementation="eager"
).eval() # 학습 전엔 평가 모드로 둠

# 번역기(토크나이저) 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_id)

# -----------------------------------------------------------------------------
# [Step 3] 학습 전 테스트 (얼마나 못하는지 확인) - 생략 가능하지만 비교용으로 좋음
# -----------------------------------------------------------------------------
# (이 부분은 보통 코드가 길어져서 생략하지만, 흐름상 "원래는 잘 못했다"를 아는 게 중요함)

# -----------------------------------------------------------------------------
# [Step 4] 데이터셋 로드 (교과서 준비)
# -----------------------------------------------------------------------------
# 네이버 뉴스 요약 데이터셋 다운로드
dataset = load_dataset('daekeun-ml/naver-news-summarization-ko')

# -----------------------------------------------------------------------------
# [Step 5] 데이터 전처리 (채팅 포맷 변환)
# -----------------------------------------------------------------------------
# Gemma는 대화형 모델이므로 데이터를 "User(본문) -> Assistant(요약)" 대화로 바꿈
def create_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": sample["document"]}, # 뉴스 본문
            {"role":"assistant", "content": sample["summary"]} # 요약 정답
        ]
    }

# 전체 데이터에 적용 및 불필요한 컬럼 삭제
train_data = dataset['train'].map(create_conversation, remove_columns=dataset['train'].features, batched=False)
test_data = dataset['test'].map(create_conversation, remove_columns=dataset['test'].features, batched=False)

# -----------------------------------------------------------------------------
# [Step 6] LoRA 설정 (수술 계획)
# -----------------------------------------------------------------------------
lora_config = LoraConfig(
    r=16,          # 어댑터 크기 (클수록 똑똑하지만 무거움, 16이 적당)
    lora_alpha=8,  # 학습 반영률 (절반 정도 힘으로 반영)
    lora_dropout=0.05,
    # Gemma 모델의 핵심 부품 4군데에 모두 어댑터를 붙임 (성능 극대화)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM" # 문장 생성 태스크
)

# -----------------------------------------------------------------------------
# [Step 7] 학습 파라미터 설정 (SFTConfig)
# -----------------------------------------------------------------------------
save_model = "gemma-text-to-sum4" # 저장할 어댑터 폴더명

args = SFTConfig(
    output_dir=save_model,
    num_train_epochs=1,             # 딱 1번만 공부시킴 (뉴스 요약은 금방 배움)
    per_device_train_batch_size=2,  # 한 번에 2문제씩 품
    gradient_accumulation_steps=4,  # 4번 모아서 채점 (총 8문제 효과) -> 학습 안정화
    gradient_checkpointing=False,   # 속도 우선 (1B 모델이라 메모리 괜찮음)
    fp16=True,                      # 16비트 가속 사용
    learning_rate=2e-4,             # 학습률 (보폭)
    optim="adamw_torch_fused",      # 최적화 도구
    logging_steps=100,              # 100번마다 기록 남김
    save_strategy="epoch",          # 다 끝나면 저장
    push_to_hub=False,              # 인터넷 업로드 안 함
    report_to='none'
)

# -----------------------------------------------------------------------------
# [Step 8] 트레이너 초기화 및 학습 시작
# -----------------------------------------------------------------------------
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_data,
    peft_config=lora_config, # 여기에 LoRA 설정 넣으면 알아서 적용됨
    processing_class=tokenizer
)

# ★★★ 여기서 시간이 제일 오래 걸림 (약 1시간 40분) ★★★
trainer.train()

# -----------------------------------------------------------------------------
# [Step 9] 어댑터 저장 및 압축
# -----------------------------------------------------------------------------
trainer.save_model() # LoRA 파일(약 20MB)만 저장됨
shutil.make_archive(save_model, 'zip', save_model) # 다운로드 편하게 압축

# =============================================================================
# [Phase 3: 병합 및 배포 - 이제 실전용 모델 만들기]
# =============================================================================

# -----------------------------------------------------------------------------
# [Step 10] 메모리 초기화 (필수!)
# -----------------------------------------------------------------------------
# 기존에 쓰던 모델 지우고 GPU 비우기 (안 하면 다음 단계에서 터짐)
del model
del trainer
torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# [Step 11] 모델 병합 (Merge)
# -----------------------------------------------------------------------------
# 1. 원본 모델 다시 로드 (이번엔 병합용 8비트 설정)
# (4비트로 병합하면 오차가 커서 보통 8비트나 16비트로 병합함)
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config
).eval()

# 2. 아까 학습한 LoRA 어댑터 장착
peft_model = PeftModel.from_pretrained(base_model, save_model)

# 3. 물리적 결합 (어댑터를 원본에 녹여버림)
merged_model = peft_model.merge_and_unload()

# -----------------------------------------------------------------------------
# [Step 12] 최종 모델 저장
# -----------------------------------------------------------------------------
# 이제 'merged_model' 폴더엔 독립적으로 실행 가능한 완벽한 모델이 들어감 (용량 약 1~2GB)
merged_model.save_pretrained("merged_model", safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained("merged_model") # 토크나이저도 꼭 같이 저장

# 결과물 압축 (이걸 다운받으면 됨)
shutil.make_archive('merged_model','zip','merged_model')

# -----------------------------------------------------------------------------
# [Step 13] 최종 테스트용 모델 로드
# -----------------------------------------------------------------------------
# 방금 만든 합체 모델("merged_model")을 불러옴
my_model = AutoModelForCausalLM.from_pretrained(
    "merged_model",
    device_map='auto',
    quantization_config=BitsAndBytesConfig(load_in_8bit=True)
).eval()

tokenizer = AutoTokenizer.from_pretrained("merged_model")

# -----------------------------------------------------------------------------
# [Step 14] 파이프라인 및 프롬프트 준비
# -----------------------------------------------------------------------------
pipe = pipeline("text-generation", model=my_model, tokenizer=tokenizer)

# 테스트 데이터 하나 꺼내기 (뉴스 본문)
test_sample = test_data[100]

# Gemma가 알아듣는 포맷(<start_of_turn>user...)으로 변환
prompt = pipe.tokenizer.apply_chat_template(
    test_sample["messages"][:1], # 정답은 빼고 질문만 줌
    tokenize=False,
    add_generation_prompt=True   # "자, 모델 너의 턴이야" 신호 추가
)

# -----------------------------------------------------------------------------
# [Step 15] 추론 (Inference)
# -----------------------------------------------------------------------------
# 멈춤 신호 설정
stop_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<end_of_turn>")]

outputs = pipe(
    prompt,
    max_new_tokens=256,
    do_sample=True,      # 약간의 창의성 허용
    temperature=0.5,     # 너무 뻔하지 않게, 너무 튀지 않게 (0.5)
    top_k=50,
    top_p=0.9,
    eos_token_id=stop_token_ids,
    no_repeat_ngram_size=2 # 앵무새 방지
)

# -----------------------------------------------------------------------------
# [Step 16] 결과 출력 (후처리)
# -----------------------------------------------------------------------------
# 전체 텍스트에서 답변 부분만 쏙 뽑아내기
final_answer = outputs[0]['generated_text'].split('<start_of_turn>model')[-1].strip()

print("\n========== [AI 요약 결과] ==========")
print(final_answer)
print("====================================")